# Terminations in Microsoft Autogen

## Why Termination Matters

We have used team to do some work, say write a story or accomplish some task. 

In [1]:
import asyncio
from autogen_ext.models.openai import OpenAIChatCompletionClient
from dotenv import load_dotenv
import os

load_dotenv()
api_key = os.getenv('OPENAI_API_KEY')
model_client = OpenAIChatCompletionClient(model='gpt-4o', api_key=api_key)

In [2]:
from autogen_agentchat.agents import AssistantAgent
add_1_agent_first = AssistantAgent(
    name = 'add_1_agent_first',
    model_client=model_client,
    system_message="Add 1 to the number, first number is 0. Give result as output"
)

add_1_agent_second = AssistantAgent(
    name = 'add_1_agent_second',
    model_client=model_client,
    system_message="Add 1 to the number. Give result as output."
)
 
add_1_agent_third = AssistantAgent(
    name = 'add_1_agent_third',
    model_client=model_client,
    system_message="Add 1 to the number. Give result as output."
)

from autogen_agentchat.teams import RoundRobinGroupChat

team = RoundRobinGroupChat(
    [add_1_agent_first, add_1_agent_second, add_1_agent_third]
)

In [3]:
from autogen_agentchat.ui import Console

# await Console(team.run_stream())

In [4]:
from autogen_agentchat.conditions import MaxMessageTermination

max_termination = MaxMessageTermination(5)

In [5]:
team = RoundRobinGroupChat(
    [add_1_agent_first, add_1_agent_second, add_1_agent_third],
    termination_condition =max_termination
)

In [6]:
from autogen_agentchat.ui import Console

await Console(team.run_stream())

---------- TextMessage (add_1_agent_first) ----------
1
---------- TextMessage (add_1_agent_second) ----------
2
---------- TextMessage (add_1_agent_third) ----------
3
---------- TextMessage (add_1_agent_first) ----------
4
---------- TextMessage (add_1_agent_second) ----------
5


TaskResult(messages=[TextMessage(id='ebe220e6-7e19-4eea-8d6b-eb5797b6be41', source='add_1_agent_first', models_usage=RequestUsage(prompt_tokens=24, completion_tokens=1), metadata={}, created_at=datetime.datetime(2026, 1, 31, 13, 24, 36, 217009, tzinfo=datetime.timezone.utc), content='1', type='TextMessage'), TextMessage(id='008261fe-0f94-49d2-bcdb-c61836a3441b', source='add_1_agent_second', models_usage=RequestUsage(prompt_tokens=29, completion_tokens=1), metadata={}, created_at=datetime.datetime(2026, 1, 31, 13, 24, 36, 668292, tzinfo=datetime.timezone.utc), content='2', type='TextMessage'), TextMessage(id='0e917cb0-2313-48e2-bcb6-0466f62f2450', source='add_1_agent_third', models_usage=RequestUsage(prompt_tokens=39, completion_tokens=1), metadata={}, created_at=datetime.datetime(2026, 1, 31, 13, 24, 37, 155050, tzinfo=datetime.timezone.utc), content='3', type='TextMessage'), TextMessage(id='ee292655-6bea-455a-93b1-9227d871b7e9', source='add_1_agent_first', models_usage=RequestUsage(pr

In [7]:
from autogen_agentchat.agents import AssistantAgent

In [21]:

agent1 = AssistantAgent(
    name = 'story_writer',
    model_client=model_client,
    system_message="Give the story about a brave knight, keep it short no more than 40 words. if critic say 'THE END' anywhere. Only output 'THE END'"
)

agent2 = AssistantAgent(
    name = 'story_critic',
    model_client=model_client,
    system_message="Continue the story and critic it with feedback. Keep it short and no more than 40 words. If it feels complete, just say 'THE END'.'"
)

In [22]:
from autogen_agentchat.conditions import TextMentionTermination


text_mention_termination = TextMentionTermination('THE END')
teamWithTextTermination = RoundRobinGroupChat(
    [agent1, agent2],
    termination_condition=text_mention_termination
)

In [25]:
from autogen_agentchat.ui import Console

await Console(teamWithTextTermination.run_stream(task = 'Write a story about a Indial girl name Jayati and her doll.'))

---------- TextMessage (user) ----------
Write a story about a Indial girl name Jayati and her doll.


---------- TextMessage (story_critic) ----------
Jayati cherished her doll, Asha, a gift from her grandmother. One stormy night, lightning struck, and Asha seemed to whisper. Together, they faced fears, turning shadows into friends, teaching Jayati that courage and imagination could light up any darkness. 

THE END.

Feedback: The story charmingly blends whimsy with a theme of courage, showcasing the power of imagination and familial bonds. Some readers might enjoy a bit more detail about their adventures, but it stands well as a comforting tale.


TaskResult(messages=[TextMessage(id='2e283ecc-9c2c-406d-8e5e-2f9f1902771f', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 1, 31, 13, 32, 18, 70770, tzinfo=datetime.timezone.utc), content='Write a story about a Indial girl name Jayati and her doll.', type='TextMessage'), TextMessage(id='227a2617-e63f-4d73-b624-cc27ec6e353d', source='story_critic', models_usage=RequestUsage(prompt_tokens=197, completion_tokens=103), metadata={}, created_at=datetime.datetime(2026, 1, 31, 13, 32, 20, 841636, tzinfo=datetime.timezone.utc), content='Jayati cherished her doll, Asha, a gift from her grandmother. One stormy night, lightning struck, and Asha seemed to whisper. Together, they faced fears, turning shadows into friends, teaching Jayati that courage and imagination could light up any darkness. \n\nTHE END.\n\nFeedback: The story charmingly blends whimsy with a theme of courage, showcasing the power of imagination and familial bonds. Some readers might enjoy a bit 

# Combining Termination Conditions

In [11]:
combined_termination = MaxMessageTermination(5) | TextMentionTermination('THE END')

from autogen_agentchat.conditions import TextMentionTermination


text_mention_termination = TextMentionTermination('THE END')
teamWithTextTermination = RoundRobinGroupChat(
    [agent1, agent2],
    termination_condition=combined_termination
)

In [12]:
from autogen_agentchat.ui import Console

await Console(teamWithTextTermination.run_stream(task = 'Write a story about a brave knight.'))

---------- TextMessage (user) ----------
Write a story about a brave knight.
---------- TextMessage (story_writer) ----------
A fearless knight faced the sorceress's curse to rescue his kingdom. Armed with a magic shield, he shattered her dark spells, freeing the land from tyranny. Celebrated by all, his name echoed in tales of valor and honor.
---------- TextMessage (story_critic) ----------
THE END


TaskResult(messages=[TextMessage(id='b8b27a55-6b15-4595-81c7-c08cd055e96d', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 1, 31, 13, 24, 39, 550344, tzinfo=datetime.timezone.utc), content='Write a story about a brave knight.', type='TextMessage'), TextMessage(id='6a8a46bf-eadf-404a-9d85-924b6ccb750a', source='story_writer', models_usage=RequestUsage(prompt_tokens=117, completion_tokens=48), metadata={}, created_at=datetime.datetime(2026, 1, 31, 13, 24, 41, 399762, tzinfo=datetime.timezone.utc), content="A fearless knight faced the sorceress's curse to rescue his kingdom. Armed with a magic shield, he shattered her dark spells, freeing the land from tyranny. Celebrated by all, his name echoed in tales of valor and honor.", type='TextMessage'), TextMessage(id='95e12ee1-986e-45bc-aae8-d5d3e84092d2', source='story_critic', models_usage=RequestUsage(prompt_tokens=185, completion_tokens=2), metadata={}, created_at=datetime.datetime(2026, 1, 31, 13, 24, 41,

# External Termination

ExternalTermination: Enables programmatic control of termination from outside the run. This is useful for UI integration (e.g., “Stop” buttons in chat interfaces).

In [13]:
import asyncio
from autogen_ext.models.openai import OpenAIChatCompletionClient
from dotenv import load_dotenv
import os

load_dotenv()
api_key = os.getenv('OPENAI_API_KEY')
model_client = OpenAIChatCompletionClient(model='gpt-4o', api_key=api_key)

In [28]:
from autogen_agentchat.agents import AssistantAgent

agent1 = AssistantAgent(
    name = 'story_writer',
    model_client=model_client,
    system_message="Give the story about a brave knight, keep it short no more than 40 words. if critic say 'THE END' anywhere. Only output 'THE END'"
)

agent2 = AssistantAgent(
    name = 'story_critic',
    model_client=model_client,
    system_message="Continue the story and critic it with feedback. Keep it short and no more than 40 words. If it feels complete, just say 'THE END'. Only output 'THE END'"
)

from autogen_agentchat.conditions import ExternalTermination
external_termination = ExternalTermination()

from autogen_agentchat.teams import RoundRobinGroupChat
team = RoundRobinGroupChat(
    [agent1, agent2],
    termination_condition= external_termination
)



In [30]:
from autogen_agentchat.ui import Console
run = asyncio.create_task(Console(team.run_stream(task = 'Write a story about a brave knight less than 40 words.')))

await asyncio.sleep(1)

external_termination.set()
await run

---------- TextMessage (user) ----------


Write a story about a brave knight less than 40 words.
---------- TextMessage (story_critic) ----------
A brave knight ventured into the dark forest, seeking the enchanted rose. Against shadows and mysterious whispers, he trusted his heart, finding the rose and restoring light to the land. Villagers called him the Knight of Hope. THE END


TaskResult(messages=[TextMessage(id='c7065188-6f62-4ba2-a2f9-6c6ca0639087', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 1, 31, 18, 22, 19, 702098, tzinfo=datetime.timezone.utc), content='Write a story about a brave knight less than 40 words.', type='TextMessage'), TextMessage(id='32ca2f3a-344f-4718-8b44-23c1f60f1c82', source='story_critic', models_usage=RequestUsage(prompt_tokens=164, completion_tokens=46), metadata={}, created_at=datetime.datetime(2026, 1, 31, 18, 22, 21, 427505, tzinfo=datetime.timezone.utc), content='A brave knight ventured into the dark forest, seeking the enchanted rose. Against shadows and mysterious whispers, he trusted his heart, finding the rose and restoring light to the land. Villagers called him the Knight of Hope. THE END', type='TextMessage')], stop_reason='External termination requested')

In [16]:
# The team is not stopping immediately but rather current agent is completing its run.

# Aborting A Team

Different from stopping a team, aborting a team will immediately stop the team and raise a CancelledError exception.

In [ ]:
from autogen_core import CancellationToken

cancellation_token = CancellationToken()

run2 = asyncio.create_task(
    Console(team.run_stream(task = 'Give a short Story about a lion atmost 40 words',cancellation_token=cancellation_token))
)

await asyncio.sleep(1)
cancellation_token.cancel()

try:
    result = await run2
except :
    print("Task Was Cancelled")

---------- TextMessage (user) ----------
Give a short Story about a lion atmost 40 words


---------- TextMessage (story_writer) ----------
A lion, strong and majestic, roamed the savanna. Facing drought, he led his pride to a hidden oasis. His courage and foresight ensured survival, and the pride thrived once more, admiring their leader's wisdom and bravery.
